## BE5210 Final Project

Binoy Patel, Sohan Lele, Eric Ouyang

### Instructions for TAs

Files to upload to Colab before running:
  - submission_models.zip
  - ensemble_params.pkl
  - truetest_data.mat

Press Run all. The notebook will unzip the models, run inference on all 3 subjects, and save predictions.mat.

Runtime: under 10 minutes on CPU, under 3 minutes on GPU. No training is performed.

Model: TensorFlow/Keras ensemble, ResNet CNN and CNN+BiGRU, 3 models per subject.

[We used AI to help debug errors in my code, explain things conceptually, and brainstorm new ideas.]

In [ ]:
#imports and unziping saved modules
import os, zipfile, pickle
import numpy as np
import scipy.io as sio
from scipy import signal as sig
from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

with zipfile.ZipFile('submission_models.zip', 'r') as zf:
    zf.extractall('.')
print('Models unzipped.')

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Models unzipped.


In [ ]:
#load normalization params
with open('ensemble_params.pkl', 'rb') as f:
    save_data = pickle.load(f)
print('Params loaded.')

Params loaded.


In [ ]:
#load TA test
mat = sio.loadmat('truetest_data.mat', simplify_cells=True)
truetest_data = mat['truetest_data']
for i, ecog in enumerate(truetest_data):
    print(f'  Subject {i+1}: {np.array(ecog).shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'truetest_data.mat'

In [ ]:
# Preprocessing: bandpass 0.5-200 Hz, remove outliers with MAD, notch filter 60/120 Hz
#then extract log band power features in 50ms windows across 11 frequency bands
def _get_bandpass(f_low, f_high, sample_rate=1000, filt_order=4):
    return sig.butter(filt_order, [f_low, f_high], btype='bandpass', fs=sample_rate, output='sos')

def filter_data(data, fs=1000):
    data = data.astype(np.float32)
    data = data - data.mean(axis=1, keepdims=True)
    channel_median = np.median(data, axis=0, keepdims=True)
    median_abs_dev = np.median(np.abs(data - channel_median), axis=0, keepdims=True) + 1e-6
    threshold = 6 * 1.4826 * median_abs_dev
    data = np.clip(data, channel_median - threshold, channel_median + threshold)
    cleaned = sig.sosfiltfilt(_get_bandpass(0.5, 200, sample_rate=fs, filt_order=4), data, axis=0)

    for notch_freq in [60, 120]:
        b, a = sig.iirnotch(notch_freq, Q=30, fs=fs)
        cleaned = sig.filtfilt(b, a, cleaned, axis=0)
    return cleaned.astype(np.float32)

FREQ_BANDS = [
    (0.5, 4), (4, 8), (8, 13), (13, 30), (30, 50),
    (50, 70), (70, 90), (90, 110), (110, 130),
    (130, 150), (150, 170)
]

def compute_band_features(filtered, fs=1000, frame_ms=50):
    win_len = int(fs * frame_ms / 1000)
    n_samples, num_channels = filtered.shape
    num_frames = n_samples // win_len
    windowed = filtered[:num_frames * win_len].reshape(num_frames, win_len, num_channels)
    feature_list = [windowed.mean(axis=1)]

    for f_low, f_high in FREQ_BANDS:
        bp_sos = _get_bandpass(f_low, f_high, sample_rate=fs, filt_order=4)
        band_signal = sig.sosfiltfilt(bp_sos, filtered, axis=0)
        band_signal = band_signal[:num_frames * win_len].reshape(num_frames, win_len, num_channels)
        feature_list.append(np.log(np.mean(band_signal ** 2, axis=1) + 1e-4))
    all_features = np.concatenate(feature_list, axis=1).astype(np.float32)

    return np.nan_to_num(all_features, nan=0.0, posinf=0.0, neginf=0.0), win_len

In [ ]:
# build sliding window sequences (30 frames of context) for the model input
# also downsamples the dg to match the feature frame rate during training
CONTEXT_FRAMES = 30
FRAME_SAMPLES = 50

def build_sequences(features, dg=None, context=CONTEXT_FRAMES):
    n_frames, F = features.shape
    pad = np.repeat(features[:1],context - 1, axis=0)
    padded = np.concatenate([pad, features], axis=0)
    X = np.lib.stride_tricks.sliding_window_view(
            padded, window_shape=context, axis=0
        ).transpose(0, 2, 1)[:n_frames]
    if dg is not None:
        Y = np.zeros((n_frames, 5), dtype=np.float32)
        for i in range(n_frames):
            s = i * FRAME_SAMPLES
            Y[i] = dg[s:s+FRAME_SAMPLES].mean(axis=0)
        return X.astype(np.float32), Y
    return X.astype(np.float32)

In [ ]:
# two model architectures: ResNet-style CNN and CNN+BiGRU hybrid
# both output 5 values, one per finger, and use Huber loss
def residual_block(inp, n_filters, kernel=3, drop_rate=0.0):
    r = layers.Conv1D(n_filters, kernel, padding='same', use_bias=False)(inp)
    r = layers.BatchNormalization()(r)
    r = layers.ReLU()(r)

    if drop_rate > 0:
        r = layers.SpatialDropout1D(drop_rate)(r)
    r = layers.Conv1D(n_filters, kernel, padding='same', use_bias=False)(r)
    r = layers.BatchNormalization()(r)

    if inp.shape[-1] != n_filters:
        inp = layers.Conv1D(n_filters, 1, padding='same', use_bias=False)(inp)
    return layers.ReLU()(layers.Add()([inp, r]))

def build_cnn(n_features):
    input_layer = keras.Input(shape=(CONTEXT_FRAMES, n_features))
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = residual_block(x, 128)
    x = residual_block(x, 192)
    x = layers.Dropout(0.2)(x)
    x = residual_block(x, 256)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(5)(x)
    model = keras.Model(input_layer, output)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=8e-4, weight_decay=1e-4, clipnorm=1.0),
        loss='huber'
    )
    return model

def build_cnn_gru(n_features):
    input_layer = keras.Input(shape=(CONTEXT_FRAMES, n_features))
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = residual_block(x, 128, drop_rate=0.15)
    x = residual_block(x, 192, drop_rate=0.15)
    x = layers.Dropout(0.35)(x)
    x = residual_block(x, 256, drop_rate=0.15)
    x = layers.Bidirectional(layers.GRU(96, return_sequences=False, dropout=0.2))(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.45)(x)
    output = layers.Dense(5)(x)
    model = keras.Model(input_layer, output)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=8e-4, weight_decay=1e-4, clipnorm=1.0),
        loss='huber'
    )
    return model


In [ ]:
# load the pre-trained models fromthe saved .keras files
models_by_subject = []
for subj in range(3):
    subject_models = [keras.models.load_model(path) for path in save_data[subj]['model_paths']]
    models_by_subject.append(subject_models)
    print(f'Subject {subj+1}: loaded {len(subject_models)} models')

In [ ]:
#run ensemble on test data, smooth predictions, and interpolate back to 1000 Hz using cubic splines
SMOOTH_SIGMAS = [4.0, 3.0, 2.5, 3.0, 4.0]
predicted_dg = []

for subj in range(3):
    print(f'Subject {subj+1}...')
    ecog = np.array(truetest_data[subj]).astype(np.float32)

    filt = filter_data(ecog)
    feat, frame_len = compute_band_features(filt)

    feat_norm = np.nan_to_num((feat - save_data[subj]['f_mean']) / save_data[subj]['f_std'], nan=0.0, posinf=0.0, neginf=0.0)
    X = build_sequences(feat_norm)

    raw_preds = np.mean([m.predict(X, batch_size=512, verbose=0) for m in models_by_subject[subj]], axis=0)
    raw_preds = raw_preds * save_data[subj]['y_std'] + save_data[subj]['y_mean']
    raw_preds = np.nan_to_num(raw_preds, nan=0.0, posinf=0.0, neginf=0.0)

    smoothed = np.stack([gaussian_filter1d(raw_preds[:, f], sigma=SMOOTH_SIGMAS[f]) for f in range(5)], axis=1)

    n_lead_samples = filt.shape[0]
    frame_centers = np.arange(smoothed.shape[0]) * frame_len + frame_len / 2
    t_out = np.arange(n_lead_samples)
    interp_dg = np.zeros((n_lead_samples, 5), dtype=np.float32)

    for i in range(5):
        t_nodes = np.concatenate([[0], frame_centers, [n_lead_samples - 1]])
        y_nodes = np.concatenate([[smoothed[0, i]], smoothed[:, i], [smoothed[-1, i]]])
        y_nodes = np.nan_to_num(y_nodes, nan=0.0, posinf=0.0, neginf=0.0)
        cs = CubicSpline(t_nodes, y_nodes, bc_type='not-a-knot')
        interp_dg[:, i] = cs(t_out).astype(np.float32)

    predicted_dg.append(interp_dg)
    print(f'  -> {interp_dg.shape}')

In [ ]:
# save predictions  as a 3x1 cell array .mat file for leaderboard submission
out = np.empty((3, 1), dtype=object)
for i in range(3):
    out[i, 0] = predicted_dg[i]

sio.savemat('predictions.mat', {'predicted_dg': out})
print('Saved predictions.mat')

v = sio.loadmat('predictions.mat')['predicted_dg']
print(f'Cell shape: {v.shape}')
for i in range(3):
    print(f'  Subject {i+1}: {v[i, 0].shape}')